In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay
)

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


In [14]:
import os

# Load dataset
df = pd.read_csv(os.path.expanduser('~/Documents/ckd-prediction-ml/data/kidney_disease.csv'))

print('='*55)
print(f'  Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print('='*55)
print(f'\nClass Distribution:')
print(df['classification'].value_counts())
print(f'\nMissing Values per Column:')
missing = df.isnull().sum()
print(missing[missing > 0])
df.head()

ParserError: Error tokenizing data. C error: Expected 25 fields in line 71, saw 26


In [16]:
import os

# Load directly from the original ARFF file
def arff_to_dataframe(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    headers = []
    rows = []
    data_start = False
    
    for line in lines:
        line = line.strip()
        if line.lower().startswith('@attribute'):
            parts = line.split()
            headers.append(parts[1].replace("'", ""))
        elif line.lower() == '@data':
            data_start = True
        elif data_start and line and not line.startswith('%'):
            rows.append(line)
    
    from io import StringIO
    import csv
    data = []
    for row in rows:
        reader = csv.reader([row])
        data.append(next(reader))
    
    df = pd.DataFrame(data, columns=headers)
    return df

arff_path = os.path.expanduser('~/Desktop/Chronic_Kidney_Disease/chronic_kidney_disease.arff')
df = arff_to_dataframe(arff_path)

print('='*55)
print(f'  Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print('='*55)
print(df['class'].value_counts())
df.head()

ValueError: 25 columns passed, passed data had 26 columns

In [18]:
import os

# Let's inspect the raw ARFF file first
arff_path = os.path.expanduser('~/Desktop/Chronic_Kidney_Disease/chronic_kidney_disease.arff')

with open(arff_path, 'r') as f:
    lines = f.readlines()

# Print headers
print("=== ATTRIBUTES ===")
headers = []
for line in lines:
    line = line.strip()
    if line.lower().startswith('@attribute'):
        parts = line.split()
        headers.append(parts[1].replace("'", ""))
        print(line)
    elif line.lower() == '@data':
        print("\n=== FIRST 5 DATA ROWS ===")
        break

# Print first 5 data rows
data_start = False
count = 0
for line in lines:
    if line.strip().lower() == '@data':
        data_start = True
        continue
    if data_start and line.strip() and not line.startswith('%'):
        print(f"Row {count+1} ({len(line.strip().split(','))} fields): {line.strip()}")
        count += 1
        if count == 5:
            break

print(f"\nTotal headers: {len(headers)}")

=== ATTRIBUTES ===
@attribute 'age' numeric
@attribute 'bp'  numeric
@attribute 'sg' {1.005,1.010,1.015,1.020,1.025}
@attribute 'al' {0,1,2,3,4,5}
@attribute 'su' {0,1,2,3,4,5}
@attribute 'rbc' {normal,abnormal}
@attribute 'pc' {normal,abnormal}
@attribute 'pcc' {present,notpresent}
@attribute 'ba' {present,notpresent}
@attribute 'bgr'  numeric
@attribute 'bu' numeric
@attribute 'sc' numeric
@attribute 'sod' numeric
@attribute 'pot' numeric
@attribute 'hemo' numeric
@attribute 'pcv' numeric
@attribute 'wbcc' numeric
@attribute 'rbcc' numeric
@attribute 'htn' {yes,no}
@attribute 'dm' {yes,no}
@attribute 'cad' {yes,no}
@attribute 'appet' {good,poor}
@attribute 'pe' {yes,no}
@attribute 'ane' {yes,no}
@attribute 'class' {ckd,notckd}

=== FIRST 5 DATA ROWS ===
Row 1 (25 fields): 48,80,1.020,1,0,?,normal,notpresent,notpresent,121,36,1.2,?,?,15.4,44,7800,5.2,yes,yes,no,good,no,no,ckd
Row 2 (25 fields): 7,50,1.020,4,0,?,normal,notpresent,notpresent,?,18,0.8,?,?,11.3,38,6000,?,no,no,no,good,no,

In [20]:
import os
import pandas as pd

arff_path = os.path.expanduser('~/Desktop/Chronic_Kidney_Disease/chronic_kidney_disease.arff')

with open(arff_path, 'r') as f:
    lines = f.readlines()

headers = []
rows = []
data_start = False

for line in lines:
    line = line.strip()
    if line.lower().startswith('@attribute'):
        parts = line.split()
        headers.append(parts[1].replace("'", ""))
    elif line.lower() == '@data':
        data_start = True
    elif data_start and line and not line.startswith('%'):
        # Replace ? with empty string (missing values)
        clean = line.replace('?', '')
        rows.append(clean.split(','))

# Fix rows that don't have exactly 25 fields
rows = [r for r in rows if len(r) == 25]

df = pd.DataFrame(rows, columns=headers)

# Replace empty strings with NaN
df = df.replace('', pd.NA)

print('='*55)
print(f'  Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print('='*55)
print(df['class'].value_counts())
print(f'\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head()

  Dataset Shape: 397 rows × 25 columns
class
ckd       248
notckd    149
Name: count, dtype: int64

Missing values:
age        9
bp        12
sg        47
al        46
su        49
rbc      150
pc        65
pcc        4
ba         4
bgr       43
bu        19
sc        17
sod       85
pot       86
hemo      52
pcv       69
wbcc     104
rbcc     129
htn        2
dm         2
cad        2
appet      1
pe         1
ane        1
dtype: int64


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48,80,1.020,1,0,<NA>,normal,notpresent,notpresent,121,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7,50,1.020,4,0,<NA>,normal,notpresent,notpresent,<NA>,...,38,6000,<NA>,no,no,no,good,no,no,ckd
2,62,80,1.010,2,3,normal,normal,notpresent,notpresent,423,...,31,7500,<NA>,no,yes,no,poor,no,yes,ckd
3,48,70,1.005,4,0,normal,abnormal,present,notpresent,117,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51,80,1.010,2,0,normal,normal,notpresent,notpresent,106,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [22]:
from scipy import stats

df_clean = df.copy()

# Step 1: Convert numeric columns from string to float
num_cols = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Step 2: Impute missing values
cat_cols = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
for col in num_cols:
    df_clean[col].fillna(df_clean[col].mean(), inplace=True)
for col in cat_cols:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

# Step 3: Remove outliers using Z-score
z_scores = np.abs(stats.zscore(df_clean[num_cols]))
before = len(df_clean)
df_clean = df_clean[(z_scores < 3).all(axis=1)]
print(f'Outliers removed: {before - len(df_clean)} | Remaining: {len(df_clean)}')

# Step 4: Encode categorical features
le = LabelEncoder()
for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col])
df_clean['class'] = df_clean['class'].str.strip().map({'ckd': 1, 'notckd': 0})

# Step 5: Scale + Split
X = df_clean.drop('class', axis=1)
y = df_clean['class']
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.30, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')
print('✅ Preprocessing complete!')

Outliers removed: 46 | Remaining: 351
Training set : 245 samples
Test set     : 106 samples
✅ Preprocessing complete!


In [24]:
# Helper evaluation function
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average=None)
    rec  = recall_score(y_te, y_pred, average=None)
    f1   = f1_score(y_te, y_pred, average=None)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='accuracy').mean()
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Test Accuracy      : {acc*100:.2f}%')
    print(f'  5-Fold CV Accuracy : {cv*100:.2f}%')
    print(f'\n{classification_report(y_te, y_pred, target_names=["Non-CKD","CKD"])}')
    return {'name': name, 'model': model, 'y_pred': y_pred,
            'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'cv': cv}

# --- Model 1: Naive Bayes ---
nb_results = evaluate_model('Naive Bayes', GaussianNB(),
                             X_train, X_test, y_train, y_test)

# --- Model 2: SVM with GridSearchCV ---
print('\nRunning GridSearchCV for SVM...')
param_grid = {'C': [0.1, 1, 10], 'gamma': [0.01, 0.1, 1], 'kernel': ['rbf']}
grid_svm = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_svm.fit(X_train, y_train)
print(f'Best SVM params: {grid_svm.best_params_}')
svm_results = evaluate_model('Optimized SVM', grid_svm.best_estimator_,
                              X_train, X_test, y_train, y_test)

# --- Model 3: Random Forest ---
rf_results = evaluate_model('Random Forest', 
                             RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
                             X_train, X_test, y_train, y_test)

results = [nb_results, svm_results, rf_results]
print('\n✅ All models trained successfully!')


  Naive Bayes
  Test Accuracy      : 94.34%
  5-Fold CV Accuracy : 95.51%

              precision    recall  f1-score   support

     Non-CKD       0.88      1.00      0.94        45
         CKD       1.00      0.90      0.95        61

    accuracy                           0.94       106
   macro avg       0.94      0.95      0.94       106
weighted avg       0.95      0.94      0.94       106


Running GridSearchCV for SVM...
Best SVM params: {'C': 1, 'gamma': 1, 'kernel': 'rbf'}

  Optimized SVM
  Test Accuracy      : 99.06%
  5-Fold CV Accuracy : 98.78%

              precision    recall  f1-score   support

     Non-CKD       0.98      1.00      0.99        45
         CKD       1.00      0.98      0.99        61

    accuracy                           0.99       106
   macro avg       0.99      0.99      0.99       106
weighted avg       0.99      0.99      0.99       106


  Random Forest
  Test Accuracy      : 100.00%
  5-Fold CV Accuracy : 98.37%

              precision  

In [25]:
import os
os.makedirs(os.path.expanduser('~/Documents/ckd-prediction-ml/outputs'), exist_ok=True)
output_path = os.path.expanduser('~/Documents/ckd-prediction-ml/outputs/')

model_names = ['Naive Bayes', 'Optimized SVM', 'Random Forest']
colors = ['#4472C4', '#70AD47', '#FFC000']

# --- Fig 1: Confusion Matrices ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
for ax, res in zip(axes, results):
    cm = confusion_matrix(y_test, res['y_pred']

SyntaxError: incomplete input (1570079432.py, line 12)

In [ ]:
import os
os.makedirs(os.path.expanduser('~/Documents/ckd-prediction-ml/outputs'), exist_ok=True)
output_path = os.path.expanduser('~/Documents/ckd-prediction-ml/outputs/')

model_names = ['Naive Bayes', 'Optimized SVM', 'Random Forest']
colors = ['#4472C4', '#70AD47', '#FFC000']

# --- Fig 1: Confusion Matrices ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
for ax, res in zip(axes, results):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Non-CKD', 'CKD'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(res['name'], fontweight='bold')
plt.tight_layout()
plt.savefig(output_path + 'confusion_matrices.png', dpi=150)
plt.show()

# --- Fig 2: Accuracy Comparison ---
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(model_names, [r['accuracy'] for r in results],
              color=colors, width=0.45, edgecolor='white')
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.80, 1.0)
for bar, res in zip(bars, results):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{res["accuracy"]*100:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(output_path + 'accuracy_comparison.png', dpi=150)
plt.show()

# --- Fig 3: Feature Importance ---
rf_model = rf_results['model']
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 6))
feat_colors = ['#C00000' if v > 0.08 else '#4472C4' for v in feat_imp.values]
ax.bar(feat_imp.index, feat_imp.values, color=feat_colors, edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(output_path + 'feature_importance.png', dpi=150)
plt.show()

# --- Fig 4: Correlation Heatmap ---
num_cols_viz = ['age', 'bp', 'sg', 'al', 'su']
corr = df_clean[num_cols_viz].corr().round(2)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, annot_kws={'size': 12})
ax.set_title('Correlation Heatmap of Key Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(output_path + 'correlation_heatmap.png', dpi=150)
plt.show()

print('✅ All charts saved to outputs folder!')

In [ ]:
# Final Summary Table
summary = pd.DataFrame({
    'Model': model_names,
    'Accuracy (%)':    [f"{r['accuracy']*100:.2f}" for r in results],
    'Precision (CKD)': [f"{r['precision'][1]:.3f}"  for r in results],
    'Recall (CKD)':    [f"{r['recall'][1]:.3f}"     for r in results],
    'F1 (CKD)':        [f"{r['f1'][1]:.3f}"         for r in results],
    'CV Accuracy (%)': [f"{r['cv']*100:.2f}"        for r in results],
})
summary.set_index('Model', inplace=True)

print('='*70)
print('  MODEL PERFORMANCE SUMMARY')
print('='*70)
print(summary.to_string())
print('='*70)
print()
print('📌 Conclusions:')
print('  ✅ Random Forest — highest accuracy, best for clinical CKD detection')
print('  ✅ Naive Bayes   — lightweight, great for early-stage screening')
print('  ✅ Optimized SVM — robust for complex non-linear patterns')
print('  ⚠️  In healthcare, Recall for CKD is critical — missing a')
print('     real CKD patient is far more costly than a false positive')